In [14]:
import time

from llama_cpp import Llama
from huggingface_hub import hf_hub_download

In [4]:
model_path = hf_hub_download(
    repo_id="ggml-org/gemma-4-E2B-it-GGUF",
    filename="gemma-4-E2B-it-Q8_0.gguf",
)

In [64]:
llm = Llama(
    model_path=model_path,
    chat_template_kwargs={"enable_thinking": False},
    n_ctx=2048,
)

llama_model_loader: loaded meta data with 44 key-value pairs and 601 tensors from /home/hansenm/.cache/huggingface/hub/models--ggml-org--gemma-4-E2B-it-GGUF/snapshots/a1dac71d3ab220618f5a7573a52acdc4baf3ae3b/gemma-4-E2B-it-Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma4
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 64
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 1.000000
llama_model_loader: - kv   5:                         general.size_label str              = 4.6B
llama_model_loader: - kv   6:                      

In [44]:
TOOLS = [
    {
      "type": "function",
      "function": {
        "name": "nevermind",
        "description": "Cancels or ignores the command.",
      }
    },
    {
      "type": "function",
      "function": {
        "name": "set_timer",
        "description": "Set a timer with a duration",
        "parameters": {
          "type": "object",
          "properties": {
            "hours":   { "type": "integer", "minimum": 0 },
            "minutes": { "type": "integer", "minimum": 0 },
            "seconds": { "type": "integer", "minimum": 0 }
          },
          "anyOf": [
            { "required": ["hours"] },
            { "required": ["minutes"] },
            { "required": ["seconds"] }
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "control_timer",
        "description": "Pause, resume, or cancel an active timer.",
        "parameters": {
          "type": "object",
          "properties": {
            "action": {
              "type": "string",
              "enum": ["pause", "resume", "cancel"]
            },
          },
          "required": ["action"]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "timer_status",
        "description": "Get the time remaining for an active timer.",
      }
    },
    {
      "type": "function",
      "function": {
        "name": "current_time",
        "description": "Get the current time.",
      }
    },
    {
      "type": "function",
      "function": {
        "name": "current_date",
        "description": "Get the current date.",
      }
    },
    {
      "type": "function",
      "function": {
        "name": "turn_on_lights",
        "description": "Turn on the lights in the current area.",
      }
    },
    {
      "type": "function",
      "function": {
        "name": "turn_off_lights",
        "description": "Turn off the lights in the current area.",
      }
    },
    {
      "type": "function",
      "function": {
        "name": "set_brightness",
        "description": "Set the brightness of the lights in the current area.",
        "parameters": {
          "type": "object",
          "properties": {
            "brightness":   { "type": "integer", "minimum": 0, "maximum": 100 },
          },
          "required": ["brightness"]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "control_media",
        "description": "Pause, resume, or skip track on media player.",
        "parameters": {
          "type": "object",
          "properties": {
            "action": {
              "type": "string",
              "enum": ["pause", "resume", "next", "previous"]
            },
          },
          "required": ["action"]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "media_volume",
        "description": "Increase, decrease, or set volume of media player.",
        "parameters": {
          "type": "object",
          "properties": {
            "action": {
              "type": "string",
              "enum": ["increase", "decrease"],
            },
            "target": {
              "type": "string", 
              "description": "Device or area name",
            },
            "level": { "type": "integer", "minimum": 0, "maximum": 100 },
          },
          "anyOf": [
            { "required": ["action"] },
            { "required": ["level"] },
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "play_music",
        "description": "Play music by artist, album, or track.",
        "parameters": {
          "type": "object",
          "properties": {
            "artist": { "type": "string" },
            "album": { "type": "string" },
            "track": { "type": "string" },
          },
          "anyOf": [
            { "required": ["artist"] },
            { "required": ["album"] },
            { "required": ["track"] },
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "weather_forecast",
        "description": "Get the current weather forecast.",
      }
    },
    {
      "type": "function",
      "function": {
        "name": "get_temperature",
        "description": "Get the current temperature of the thermostat.",
      }
    },
    {
      "type": "function",
      "function": {
        "name": "add_todo",
        "description": "Add a task to the todo list.",
        "parameters": {
          "type": "object",
          "properties": {
            "item": { "type": "string" },
          },
          "anyOf": [
            { "required": ["item"] },
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "complete_todo",
        "description": "Mark a task as completed on the todo list.",
        "parameters": {
          "type": "object",
          "properties": {
            "item": { "type": "string" },
          },
          "anyOf": [
            { "required": ["item"] },
          ]
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "add_to_shopping_list",
        "description": "Add an item to the shopping list.",
        "parameters": {
          "type": "object",
          "properties": {
            "item": { "type": "string" },
          },
          "anyOf": [
            { "required": ["item"] },
          ]
        }
      }
    },
]

In [59]:
import json
import re

TOOL_CALL_RE = re.compile(
    r"<\|tool_call>call:([a-zA-Z0-9_]+)\{(.*?)\}<tool_call\|>",
    re.DOTALL,
)

def _normalize_gemma_tool_text(text: str) -> str:
    return (
        text
        .replace('<|"|>', '"')
        .replace("<|'|>", "'")
    )

def _parse_value(value: str):
    value = _normalize_gemma_tool_text(value.strip())

    if len(value) >= 2 and value[0] == '"' and value[-1] == '"':
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value[1:-1]

    if re.fullmatch(r"-?\d+", value):
        return int(value)

    if re.fullmatch(r"-?\d+\.\d+", value):
        return float(value)

    lowered = value.lower()
    if lowered == "true":
        return True
    if lowered == "false":
        return False
    if lowered == "null":
        return None

    return value

def _split_args(raw_args: str) -> list[str]:
    parts = []
    buf = []
    in_string = False
    escape = False

    for ch in raw_args:
        if escape:
            buf.append(ch)
            escape = False
            continue

        if ch == "\\":
            buf.append(ch)
            escape = True
            continue

        if ch == '"':
            buf.append(ch)
            in_string = not in_string
            continue

        if ch == "," and not in_string:
            part = "".join(buf).strip()
            if part:
                parts.append(part)
            buf.clear()
            continue

        buf.append(ch)

    part = "".join(buf).strip()
    if part:
        parts.append(part)

    return parts

def parse_tool_calls(text: str) -> list[tuple[str, dict]]:
    text = _normalize_gemma_tool_text(text)
    calls = []

    for match in TOOL_CALL_RE.finditer(text):
        name = match.group(1)
        raw_args = match.group(2).strip()
        args = {}

        if raw_args:
            for part in _split_args(raw_args):
                key, value = part.split(":", 1)
                args[key.strip()] = _parse_value(value)

        calls.append((name, args))

    return calls

In [62]:
text = "turn on the lights and tell me the weather"

start_time = time.monotonic()
response = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": text}
    ],
    tools=TOOLS,
    temperature=0,
    top_p=1.0,
    max_tokens=32,
    tool_choice="auto",
)
end_time = time.monotonic()
print(end_time - start_time)

response

Llama.generate: 659 prefix-match hit, remaining 14 prompt tokens to eval
llama_perf_context_print:        load time =    3320.91 ms
llama_perf_context_print: prompt eval time =     178.12 ms /    14 tokens (   12.72 ms per token,    78.60 tokens per second)
llama_perf_context_print:        eval time =    1365.83 ms /    18 runs   (   75.88 ms per token,    13.18 tokens per second)
llama_perf_context_print:       total time =    1554.21 ms /    32 tokens
llama_perf_context_print:    graphs reused =         17


1.5597438429977046


{'id': 'chatcmpl-adb56e36-e482-4925-a522-a75d253f705b',
 'object': 'chat.completion',
 'created': 1778101562,
 'model': '/home/hansenm/.cache/huggingface/hub/models--ggml-org--gemma-4-E2B-it-GGUF/snapshots/a1dac71d3ab220618f5a7573a52acdc4baf3ae3b/gemma-4-E2B-it-Q8_0.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': '<|tool_call>call:turn_on_lights{}<tool_call|><|tool_call>call:weather_forecast{}<tool_call|>'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 673, 'completion_tokens': 18, 'total_tokens': 691}}

In [63]:
parse_tool_calls(response["choices"][0]["message"]["content"])

[('turn_on_lights', {}), ('weather_forecast', {})]